# Data Prep

In [ ]:
!pip install opencv-python numpy pandas pillow

In [ ]:
!pip install google-generativeai pandas opencv-python thefuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 56.3 MB/s eta 0:00:00


In [ ]:
!pip install thefuzz

In [ ]:
import cv2
import numpy as np
import os
import zipfile
from concurrent.futures import ProcessPoolExecutor

class ArgyleDataPrep:
    def __init__(self, zip_path="image.zip"):
        self.zip_path = zip_path
        self.extract_folder = "extracted_images"
        self.processed_folder = "processed_posters"

        for folder in [self.extract_folder, self.processed_folder]:
            os.makedirs(folder, exist_ok=True)

        self._unzip_images()

    def _unzip_images(self):
        if not os.path.exists(self.zip_path):
            print(f"Error: {self.zip_path} not found.")
            return
        with zipfile.ZipFile(self.zip_path, 'r') as zip_ref:
            zip_ref.extractall(self.extract_folder)
        print(f"Extraction complete.")

    def deskew_image(self, img):
        _, thresh = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
        pixels = np.where(thresh > 0)

        if len(pixels[0]) < 20:
            return img
        coords = np.column_stack((pixels[1], pixels[0])).astype(np.float32)
        rect = cv2.minAreaRect(coords)
        angle = rect[-1]

        if angle < -45:
            angle = -(90 + angle)
        else:
            angle = -angle
        if abs(angle) < 0.5:
            return img


        (h, w) = img.shape[:2]
        center = (w // 2, h // 2)
        M = cv2.getRotationMatrix2D(center, angle, 1.0)
        rotated = cv2.warpAffine(
            img, M, (w, h),
            flags=cv2.INTER_CUBIC,
            borderMode=cv2.BORDER_REPLICATE
        )
        return rotated

    def optimize_for_llm(self, file_info):
        image_path, save_name = file_info

        img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            return None


        h_orig, w_orig = img.shape[:2]
        scale = 1500 / w_orig
        img = cv2.resize(img, (1500, int(h_orig * scale)), interpolation=cv2.INTER_CUBIC)
        img = self.deskew_image(img)

        clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
        img = clahe.apply(img)
        img = cv2.fastNlMeansDenoising(img, None, 10, 7, 21)

        save_path = os.path.join(self.processed_folder, save_name)
        cv2.imwrite(save_path, img)
        return save_name

if __name__ == "__main__":
    prep = ArgyleDataPrep(zip_path="image.zip")
    tasks = []
    for root, dirs, files in os.walk(prep.extract_folder):
        for file in files:
            if file.lower().endswith(('.jpg', '.jpeg', '.png')):
                tasks.append((os.path.join(root, file), file))

    with ProcessPoolExecutor() as executor:
        results = list(executor.map(prep.optimize_for_llm, tasks))

    success_count = len([r for r in results if r])
    print(f"{success_count} posters straightened and enhanced")

📦 Unzipping image.zip...
Extraction complete.
30 posters straightened and enhanced


# AI extraction

gemini pipeline, output: hackathon.csv

In [ ]:
api_key = 'make_your_own_api_key :) '

In [ ]:
import google.generativeai as genai
import json
import os
import pandas as pd
import time
import threading
from concurrent.futures import ThreadPoolExecutor

genai.configure(api_key=api_key)

generation_config = {
    "temperature": 0.1,
    "response_mime_type": "application/json",
    "response_schema": {
        "type": "array",
        "items": {
            "type": "object",
            "properties": {
                "act_name": {"type": "string"},
                "act_description": {"type": "string"},
                "performance_date": {"type": "string"}
            },
            "required": ["act_name", "act_description"]
        }
    }
}

model = genai.GenerativeModel(model_name='gemini-2.5-pro', generation_config=generation_config)

progress_lock = threading.Lock()
completed_count = 0

def get_expert_prompt(accession_id):
    return f"""
    PERSONA: Expert Theatre Historian and Archival Transcriber.

    TASK: Extract the full performing act details from this Argyle Theatre poster (ID: {accession_id}).

    CONTEXT:
    I need the 'Flavor' of the poster.
    - 'act_name': The large, prominent name of the performer.
    - 'act_description': The specific descriptive phrase found underneath the name.
      Example: If the poster says 'The Jolly Juggling Jesters', extract that ENTIRE phrase.
      Do not simplify it to just 'Juggler'. Capture the artistic flair.

    RULES:
    1. Locate the 'Performance Date' from the top header and include it for every row.
    2. Transcription must be high-fidelity.
    3. Fix obvious OCR typos but keep the historical wording exactly as it appears.

    FORMAT: Return a flat JSON array.
    """


# 3. EXTRACTION LOGIC

def extract_poster_data(filename, total_files):
    global completed_count
    path = os.path.join("processed_posters", filename)
    accession_id = filename.replace("NFA", "").replace("-", ".").split(".")[0]

    safety_settings = [
        {"category": "HARM_CATEGORY_HARASSMENT", "threshold": "BLOCK_NONE"},
        {"category": "HARM_CATEGORY_HATE_SPEECH", "threshold": "BLOCK_NONE"},
        {"category": "HARM_CATEGORY_SEXUALLY_EXPLICIT", "threshold": "BLOCK_NONE"},
        {"category": "HARM_CATEGORY_DANGEROUS_CONTENT", "threshold": "BLOCK_NONE"},
    ]

    for attempt in range(3):
        try:
            img_file = genai.upload_file(path=path)
            prompt = get_expert_prompt(accession_id)
            response = model.generate_content([prompt, img_file], safety_settings=safety_settings)

            content = response.text.strip()
            if content.startswith("```"):
                content = content.split("```")[1]
                if content.startswith("json"):
                    content = content[4:].strip()

            acts_list = json.loads(content)
            genai.delete_file(img_file.name)

            for act in acts_list:
                act['accession_id'] = accession_id
                act['source_file'] = filename

            with progress_lock:
                completed_count += 1
                print(f" [{completed_count}/{total_files}] Processed: {filename}")

            return acts_list

        except Exception as e:
            time.sleep((attempt + 1) * 3)
    return []
def run_extraction_pipeline():
    processed_folder = "processed_posters"
    if not os.path.exists(processed_folder):
        print("Error: processed_posters folder missing.")
        return

    files = [f for f in os.listdir(processed_folder) if f.lower().endswith(('.jpg', '.png'))]
    total = len(files)

    print(f"Starting Historical Transcription for {total} posters...")

    all_rows = []
    with ThreadPoolExecutor(max_workers=3) as executor:
        futures = [executor.submit(extract_poster_data, f, total) for f in files]
        for future in futures:
            all_rows.extend(future.result())

    if all_rows:
        df = pd.DataFrame(all_rows)
        final_cols = ['accession_id', 'performance_date', 'act_name', 'act_description', 'source_file']
        df = df.reindex(columns=final_cols)

        output_file = "argyle_master_archive.csv"
        df.to_csv(output_file, index=False, encoding='utf-8-sig')

    else:
        print("\n Extraction failed. No data recovered.")

if __name__ == "__main__":
    run_extraction_pipeline()

In [ ]:
import pandas as pd


df1 = pd.read_csv('argyle_second_output.csv')
df2 = pd.read_csv('Poster_tagging - Combined.csv')


df1 = df1.rename(columns={
    'act_name': 'performer',
    'act_description': 'act_name'
})


print("Updated Columns in df1:", df1.columns.tolist())
print("Updated Columns in df2:", df2.columns.tolist())

Updated Columns in df1: ['accession_id', 'performance_date', 'performer', 'act_name', 'source_file']
Updated Columns in df2: ['file', 'image', 'venue_name', 'venue_location', 'date', 'act_name', 'performer', 'description', 'billing_notes', 'ticket_prices']


In [ ]:
import pandas as pd


df1['source_file_clean'] = df1['source_file'].str.replace('.jpg', '', case=False, regex=False).str.strip()


df2['file_clean'] = df2['file'].str.strip()


df1_unique = set(df1['source_file_clean'].dropna().unique())
df2_unique = set(df2['file_clean'].dropna().unique())


missing_in_df1 = df2_unique - df1_unique
extra_in_df1 = df1_unique - df2_unique

print(f"Total unique files in df1 (Extracted): {len(df1_unique)}")
print(f"Total unique files in df2 (Reference): {len(df2_unique)}")

print(f"Number of files successfully matched: {len(df1_unique.intersection(df2_unique))}")
print(f"Number of files missing in df1: {len(missing_in_df1)}")

if len(missing_in_df1) > 0:
    print("\nFiles in archive but NOT extracted by AI:")
    print(sorted(list(missing_in_df1)))

if len(extra_in_df1) > 0:
    print("\nFiles AI extracted that aren't in the reference list:")
    print(sorted(list(extra_in_df1)))

Total unique files in df1 (Extracted): 24
Total unique files in df2 (Reference): 30
---
Number of files successfully matched: 24
Number of files missing in df1: 6

Files in archive but NOT extracted by AI:
['NFA178R12-10', 'NFA178R12-17', 'NFA178R12-204', 'NFA178R12-263', 'NFA178R12-28', 'NFA178R12-9']


In [ ]:
import os
import cv2
import pandas as pd

processed_folder = "processed_posters"
failed_files = ['NFA178R12-10.jpg', 'NFA178R12-17.jpg', 'NFA178R12-204.jpg',
                'NFA178R12-263.jpg', 'NFA178R12-28.jpg', 'NFA178R12-9.jpg']

audit_data = []

for file in os.listdir(processed_folder):
    if file.lower().endswith(('.jpg', '.png')):
        path = os.path.join(processed_folder, file)
        img = cv2.imread(path)

        if img is not None:
            h, w = img.shape[:2]
            file_size_kb = os.path.getsize(path) / 1024
            status = "Failed" if file in failed_files else "Success"

            audit_data.append({
                "filename": file,
                "width": w,
                "height": h,
                "aspect_ratio": round(h/w, 2),
                "size_kb": round(file_size_kb, 2),
                "status": status
            })

audit_df = pd.DataFrame(audit_data)



print(audit_df.groupby("status")[["aspect_ratio", "size_kb"]].mean())



print(audit_df[audit_df['status'] == "Failed"])

--- Average Metrics: Success vs Failure ---
         aspect_ratio      size_kb
status                            
Failed        2.89000  1812.640000
Success       2.89875  2123.556667

--- Detailed View of Failed Files ---
             filename  width  height  aspect_ratio  size_kb  status
11   NFA178R12-28.jpg   1500    4208          2.81  1383.60  Failed
19  NFA178R12-204.jpg   1500    4501          3.00  2669.30  Failed
21   NFA178R12-10.jpg   1500    4345          2.90  1393.72  Failed
22    NFA178R12-9.jpg   1500    4315          2.88  1594.07  Failed
24   NFA178R12-17.jpg   1500    4194          2.80  1357.52  Failed
28  NFA178R12-263.jpg   1500    4432          2.95  2477.63  Failed


In [ ]:
import cv2
import numpy as np
import os
import pandas as pd

def get_image_metrics(image_path):
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None

    blur_score = cv2.Laplacian(img, cv2.CV_64F).var()

    contrast_score = img.std()

    brightness = np.mean(img)

    return blur_score, contrast_score, brightness


failed_files = ['NFA178R12-10.jpg', 'NFA178R12-17.jpg', 'NFA178R12-204.jpg',
                'NFA178R12-263.jpg', 'NFA178R12-28.jpg', 'NFA178R12-9.jpg']

processed_folder = "processed_posters"
results = []

for file in os.listdir(processed_folder):
    if file.lower().endswith(('.jpg', '.png')):
        path = os.path.join(processed_folder, file)
        metrics = get_image_metrics(path)
        if metrics:
            is_failed = file in failed_files
            results.append({
                "file": file,
                "blur_score": metrics[0],
                "contrast": metrics[1],
                "brightness": metrics[2],
                "outcome": "Rejected" if is_failed else "Success"
            })


df_metrics = pd.DataFrame(results)

analysis = df_metrics.groupby("outcome")[["blur_score", "contrast", "brightness"]].mean()

print("--- Forensic Image Analysis Results ---")
print(analysis)


if "Rejected" in analysis.index and "Success" in analysis.index:
    contrast_diff = ((analysis.loc['Success', 'contrast'] / analysis.loc['Rejected', 'contrast']) - 1) * 100
    blur_diff = ((analysis.loc['Success', 'blur_score'] / analysis.loc['Rejected', 'blur_score']) - 1) * 100


    print(f"1. Contrast Gap: Success files have {contrast_diff:.1f}% higher contrast.")
    print(f"2. Sharpness Gap: Success files are {blur_diff:.1f}% sharper (less blurry).")

--- Forensic Image Analysis Results ---
          blur_score   contrast  brightness
outcome                                    
Rejected  347.327722  64.931683  152.166712
Success   674.273302  65.640716  151.527777

--- Data Science Insights for Slides ---
1. Contrast Gap: Success files have 1.1% higher contrast.
2. Sharpness Gap: Success files are 94.1% sharper (less blurry).


In [ ]:
import pandas as pd


df1['file_id'] = df1['source_file'].str.replace('.jpg', '', case=False, regex=False).str.strip()
df2['file_id'] = df2['file'].str.strip()


ai_counts = df1.groupby('file_id').size().reset_index(name='ai_act_count')
manual_counts = df2.groupby('file_id').size().reset_index(name='manual_act_count')


comparison_df = pd.merge(manual_counts, ai_counts, on='file_id', how='inner')


comparison_df['difference'] = comparison_df['ai_act_count'] - comparison_df['manual_act_count']


perfect_matches = comparison_df[comparison_df['difference'] == 0]
missed_some = comparison_df[comparison_df['difference'] < 0]
found_extra = comparison_df[comparison_df['difference'] > 0]

print(f"--- Extraction Completeness Analysis (n=24) ---")
print(f"Posters with 100% matching act counts: {len(perfect_matches)}")
print(f"Posters where AI missed acts: {len(missed_some)}")
print(f"Posters where AI found extra acts: {len(found_extra)}")

print("\n--- Detailed Comparison Table ---")
print(comparison_df[['file_id', 'manual_act_count', 'ai_act_count', 'difference']])

total_manual_acts = comparison_df['manual_act_count'].sum()
total_ai_acts = comparison_df['ai_act_count'].sum()
recall_accuracy = (total_ai_acts / total_manual_acts) * 100

print(f"\nOverall Extraction Recall: {recall_accuracy:.1f}%")

--- Extraction Completeness Analysis (n=24) ---
Posters with 100% matching act counts: 4
Posters where AI missed acts: 11
Posters where AI found extra acts: 9

--- Detailed Comparison Table ---
          file_id  manual_act_count  ai_act_count  difference
0   NFA178R12-103                10            10           0
1   NFA178R12-104                11            10          -1
2    NFA178R12-16                12             9          -3
3    NFA178R12-18                 8            10           2
4   NFA178R12-205                10             9          -1
5   NFA178R12-207                11            10          -1
6   NFA178R12-208                10            13           3
7   NFA178R12-209                11            10          -1
8   NFA178R12-211                10            12           2
9   NFA178R12-215                12             9          -3
10  NFA178R12-220                 9            14           5
11  NFA178R12-258                 7             9           2
